# CMAPSS FD003 - Exploratory Data Analysis

This notebook performs a quick, repeatable EDA workflow for the FD003 subset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)

In [ ]:
DATA_DIR = Path("CMAPSSData")
subset = "FD003"

train_path = DATA_DIR / f"train_{subset}.txt"
test_path = DATA_DIR / f"test_{subset}.txt"
rul_path = DATA_DIR / f"RUL_{subset}.txt"

columns = ["unit_nr", "time_cycles", "op_setting_1", "op_setting_2", "op_setting_3"] + [f"sensor_{i}" for i in range(1, 22)]

train_df = pd.read_csv(train_path, sep=r"\s+", header=None)
test_df = pd.read_csv(test_path, sep=r"\s+", header=None)
rul_df = pd.read_csv(rul_path, sep=r"\s+", header=None)

train_df.columns = columns
test_df.columns = columns
rul_df.columns = ["RUL"]

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"RUL shape: {rul_df.shape}")

In [ ]:
train_df.head()

In [ ]:
summary_df = pd.DataFrame({
    "dtype": train_df.dtypes.astype(str),
    "missing": train_df.isna().sum(),
    "nunique": train_df.nunique()
})

summary_df

In [ ]:
max_cycle_by_unit = train_df.groupby("unit_nr")["time_cycles"].max().rename("max_cycle")
train_rul = train_df.join(max_cycle_by_unit, on="unit_nr")
train_rul["RUL"] = train_rul["max_cycle"] - train_rul["time_cycles"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_rul["RUL"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Train RUL Distribution")
axes[0].set_xlabel("RUL")

sns.boxplot(x=train_rul["RUL"], ax=axes[1])
axes[1].set_title("Train RUL Boxplot")
axes[1].set_xlabel("RUL")

plt.tight_layout()

In [ ]:
sensor_cols = [c for c in train_df.columns if c.startswith("sensor_")]
sensor_variance = train_df[sensor_cols].var().sort_values(ascending=False)
sensor_variance.head(10)

In [ ]:
top_sensors = sensor_variance.head(3).index.tolist()
sample_units = train_df["unit_nr"].drop_duplicates().sample(min(5, train_df["unit_nr"].nunique()), random_state=42)
sample_df = train_df[train_df["unit_nr"].isin(sample_units)]

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(12, 3 * len(top_sensors)), sharex=True)
if len(top_sensors) == 1:
    axes = [axes]

for i, sensor in enumerate(top_sensors):
    sns.lineplot(data=sample_df, x="time_cycles", y=sensor, hue="unit_nr", ax=axes[i], legend=(i == 0))
    axes[i].set_title(f"{sensor} trend for sampled engines")

plt.tight_layout()

In [ ]:
top_corr_sensors = sensor_variance.head(12).index.tolist()
corr = train_df[top_corr_sensors].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Top-Variance Sensors")
plt.tight_layout()

## Notes

- Compare sensor variance and correlation patterns across subsets.
- Reuse the same workflow to keep comparisons fair and reproducible.